In [1]:

%pip install pandas numpy spacy matplotlib plotly scikit-learn beautifulsoup4 nltk fastopic optuna optuna-dashboard

Note: you may need to restart the kernel to use updated packages.


In [2]:

import random
import re
import time

import matplotlib.pyplot as plt
import numpy as np
import optuna
import pandas as pd
from fastopic import FASTopic
from gensim.corpora import Dictionary
from gensim.models import CoherenceModel
from wordcloud import WordCloud

## FASTopic

In [3]:
# Carga de datos
test = pd.read_csv("data/research-articles/test.csv")
train = pd.read_csv("data/research-articles/train.csv")

display(test.head())
display(train.head())

,ID,TITLE,ABSTRACT
0,20973,Closed-form Marginal Likelihood in Gamma-Poiss...,We present novel understandings of the Gamma...
1,20974,Laboratory mid-IR spectra of equilibrated and ...,Meteorites contain minerals from Solar Syste...
2,20975,Case For Static AMSDU Aggregation in WLANs,Frame aggregation is a mechanism by which mu...
3,20976,The $Gaia$-ESO Survey: the inner disk intermed...,Milky Way open clusters are very diverse in ...
4,20977,Witness-Functions versus Interpretation-Functi...,Proving that a cryptographic protocol is cor...


,ID,TITLE,ABSTRACT,Computer Science,Physics,Mathematics,Statistics,Quantitative Biology,Quantitative Finance
0,1,Reconstructing Subject-Specific Effect Maps,Predictive models allow subject-specific inf...,1,0,0,0,0,0
1,2,Rotation Invariance Neural Network,Rotation invariance and translation invarian...,1,0,0,0,0,0
2,3,Spherical polyharmonics and Poisson kernels fo...,We introduce and develop the notion of spher...,0,0,1,0,0,0
3,4,A finite element approximation for the stochas...,The stochastic Landau--Lifshitz--Gilbert (LL...,0,0,1,0,0,0
4,5,Comparative study of Discrete Wavelet Transfor...,Fourier-transform infra-red (FTIR) spectra o...,1,0,0,1,0,0


In [4]:
train.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 20972 entries, 0 to 20971
Data columns (total 9 columns):
 #   Column                Non-Null Count  Dtype 
---  ------                --------------  ----- 
 0   ID                    20972 non-null  int64 
 1   TITLE                 20972 non-null  object
 2   ABSTRACT              20972 non-null  object
 3   Computer Science      20972 non-null  int64 
 4   Physics               20972 non-null  int64 
 5   Mathematics           20972 non-null  int64 
 6   Statistics            20972 non-null  int64 
 7   Quantitative Biology  20972 non-null  int64 
 8   Quantitative Finance  20972 non-null  int64 
dtypes: int64(7), object(2)
memory usage: 1.4+ MB


In [5]:
train.dtypes

ID                       int64
TITLE                   object
ABSTRACT                object
Computer Science         int64
Physics                  int64
Mathematics              int64
Statistics               int64
Quantitative Biology     int64
Quantitative Finance     int64
dtype: object

In [6]:
train_small = train.sample(n=10, random_state=42)
documents = list(train_small.ABSTRACT.values)  # utilizaremos solo la columan Abstract como los documentos
print(documents[0])  # como ejemplo imprimimos el primer documento

  Layer normalization is a recently introduced technique for normalizing the
activities of neurons in deep neural networks to improve the training speed and
stability. In this paper, we introduce a new layer normalization technique
called Dynamic Layer Normalization (DLN) for adaptive neural acoustic modeling
in speech recognition. By dynamically generating the scaling and shifting
parameters in layer normalization, DLN adapts neural acoustic models to the
acoustic variability arising from various factors such as speakers, channel
noises, and environments. Unlike other adaptive acoustic models, our proposed
approach does not require additional adaptation data or speaker information
such as i-vectors. Moreover, the model size is fixed as it dynamically
generates adaptation parameters. We apply our proposed DLN to deep
bidirectional LSTM acoustic models and evaluate them on two benchmark datasets
for large vocabulary ASR experiments: WSJ and TED-LIUM release 2. The
experimental results s

In [7]:
clean_text = documents
tokenized_texts = [doc.split() for doc in clean_text]
print(tokenized_texts[0])
dictionary = Dictionary(tokenized_texts)
dictionary.filter_extremes(no_below=5, no_above=0.8)

['Layer', 'normalization', 'is', 'a', 'recently', 'introduced', 'technique', 'for', 'normalizing', 'the', 'activities', 'of', 'neurons', 'in', 'deep', 'neural', 'networks', 'to', 'improve', 'the', 'training', 'speed', 'and', 'stability.', 'In', 'this', 'paper,', 'we', 'introduce', 'a', 'new', 'layer', 'normalization', 'technique', 'called', 'Dynamic', 'Layer', 'Normalization', '(DLN)', 'for', 'adaptive', 'neural', 'acoustic', 'modeling', 'in', 'speech', 'recognition.', 'By', 'dynamically', 'generating', 'the', 'scaling', 'and', 'shifting', 'parameters', 'in', 'layer', 'normalization,', 'DLN', 'adapts', 'neural', 'acoustic', 'models', 'to', 'the', 'acoustic', 'variability', 'arising', 'from', 'various', 'factors', 'such', 'as', 'speakers,', 'channel', 'noises,', 'and', 'environments.', 'Unlike', 'other', 'adaptive', 'acoustic', 'models,', 'our', 'proposed', 'approach', 'does', 'not', 'require', 'additional', 'adaptation', 'data', 'or', 'speaker', 'information', 'such', 'as', 'i-vectors.

In [8]:
def coherence_from_topics(topic_words, tokenized_texts, dictionary):
    if not topic_words:
        return -1e6
    try:
        cm = CoherenceModel(topics=topic_words, texts=tokenized_texts, dictionary=dictionary, coherence='c_v')
        return float(cm.get_coherence())
    except Exception as e:
        print("Error coherence:", e)
        return -1e6

In [9]:
def get_top_words(model, num_top_words=10):
    if num_top_words is None:
        num_top_words = num_top_words
    return model.get_top_words(num_top_words)

In [13]:
def objective(trial):
    """
    Construye un FASTopic con los hiperparámetros sugeridos, entrena (fit_transform)
    y devuelve coherencia c_v (a maximizar).
    """
    # Hiperparámetros a buscar
    n_topics = trial.suggest_int("num_topics", 10, 80)  # rango típico
    DT_alpha = trial.suggest_categorical("DT_alpha", [1.0, 5.0, 10.0, 15.0])
    normalize_embeddings = trial.suggest_categorical("normalize_embeddings", [True, False])
    learning_rate = trial.suggest_loguniform("learning_rate", 1e-3, 1e-1)
    epochs = trial.suggest_int("epochs", 20, 200)

    # build model (constructor flexible según versión)
    try:
        # FASTopic puede aceptar: FASTopic(num_topics, preprocess=..., doc_embed_model=..., normalize_embeddings=...)
        # Aquí usamos constructor con num_topics y algunos flags; si tu versión tiene firma distinta, se adapta por try/except
        model = FASTopic(num_topics=n_topics,
                         normalize_embeddings=normalize_embeddings,
                         DT_alpha=DT_alpha,
                         verbose=False)
    except TypeError:
        # fallback: construir sin algunos argumentos
        model = FASTopic(num_topics=n_topics, verbose=False)

    # Entrenar (fit_transform devuelve top_words y doc_topic_dist según README)

    t0 = time.time()
    top_words_out, doc_topic_dist = model.fit_transform(clean_text, learning_rate=learning_rate, epochs=epochs)
    elapsed = time.time() - t0

    if len(top_words_out) == 0:
        trial.set_user_attr("n_topics_found", 0)
        return -1e6

    topics_for_gensim = get_top_words(model)
    print("Top words found:", topics_for_gensim)

    # calcular coherencia c_v
    coh = coherence_from_topics(topics_for_gensim, tokenized_texts, dictionary)

    # guardar attrs útiles
    trial.set_user_attr("elapsed_s", elapsed)
    trial.set_user_attr("n_topics_found", len(topics_for_gensim))

    return float(coh)


In [14]:
RANDOM_SEED = 42
random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)
def run_optuna_fastopic(n_trials=20):
    sampler = optuna.samplers.TPESampler(seed=RANDOM_SEED)
    study = optuna.create_study(direction="maximize", sampler=sampler)
    print("Iniciando Optuna para FASTopic. n_trials =", n_trials)
    study.optimize(objective, n_trials=n_trials)
    print("Mejor coherencia:", study.best_value)
    print("Mejores hiperparámetros:", study.best_params)
    return study

In [15]:
study = run_optuna_fastopic(n_trials=20)

[I 2025-12-05 19:06:30,840] A new study created in memory with name: no-name-eb22283a-cc97-49a1-9fc8-20e091c70fa2
C:\Users\zeldan\AppData\Local\Temp\ipykernel_12764\3245100855.py:10: FutureWarning:

suggest_loguniform has been deprecated in v3.0.0. This feature will be removed in v6.0.0. See https://github.com/optuna/optuna/releases/tag/v3.0.0. Use suggest_float(..., log=True) instead.



Iniciando Optuna para FASTopic. n_trials = 20


parsing texts: 100%|██████████| 10/10 [00:00<00:00, 6651.29it/s]
C:\Users\zeldan\anaconda3\envs\tfg-topics2\Lib\site-packages\sklearn\feature_extraction\text.py:517: UserWarning:

The parameter 'token_pattern' will not be used since 'tokenizer' is not None'

Training FASTopic: 100%|██████████| 128/128 [00:14<00:00,  9.07it/s]
[I 2025-12-05 19:06:46,977] Trial 0 finished with value: -1000000.0 and parameters: {'num_topics': 36, 'DT_alpha': 1.0, 'normalize_embeddings': True, 'learning_rate': 0.05399484409787434, 'epochs': 128}. Best is trial 0 with value: -1000000.0.


Top words found: ['popular algorithmic modularity greedy quantitative performance clear counting insight states', 'cases applications linear orders author classical unique suggestion yangian form', 'proposed approach limit improve large adaptive sparsity achieved based modularity', 'gravitational conjectural identification result algebraic yau correspondence unable twisted formulation', 'model new data require generating based time rewards optimal reward', 'code nonlinear pulsedyn non accessible morse fermi body specialists toda', 'terms perturbation explicitly dual paper infty duality supersymmetric large presentation', 'log include pettie round coloring lll currently enlightening checkable essentially', 'degree bounded arxiv improvement rounds log distributed best bound include', 'adaptive deep vectors called variability information benchmark networks adapts recognition', 'acoustic dynamically models layer dln normalization neural technique environments adaptation', 'non pulsedyn non

Training FASTopic: 100%|██████████| 114/114 [00:02<00:00, 39.43it/s]
[I 2025-12-05 19:06:51,613] Trial 1 finished with value: -1000000.0 and parameters: {'num_topics': 60, 'DT_alpha': 5.0, 'normalize_embeddings': False, 'learning_rate': 0.0040596116104843075, 'epochs': 114}. Best is trial 0 with value: -1000000.0.


...

Top words found: ['acoustic models dln layer neural normalization dynamically adaptive proposed new', 'missing clustering mixture outliers processed distributions allow learning variables treatment', 'coloring lcl sqrt chung lll complexity pettie randomized round known', 'carleson bounded degree obtained infty mathbf mathbb operators fixed operator', 'values gives treatment handle approximation latent having scale developed normal', 'kernel continuous define lies multidimensional zygmund let truncations extends operator', 'conjectural holds presentation explicit quantum formulation enumerative identification formulated quantized', 'code nonlinear systems accessible body pasta tsingou dynamics integrable lennard', 'diagonal markov random ising approaches response formulate constructed matching computation', 'popular modularity insight greedy algorithmic maximization approach presumably risk provides', 'explicitly duality presented supersymmetric dual suggestion perturbation background a

Training FASTopic: 100%|██████████| 148/148 [00:04<00:00, 30.58it/s]
[I 2025-12-05 19:09:51,570] Trial 18 finished with value: -1000000.0 and parameters: {'num_topics': 20, 'DT_alpha': 5.0, 'normalize_embeddings': False, 'learning_rate': 0.018158532922746568, 'epochs': 148}. Best is trial 0 with value: -1000000.0.


Top words found: ['modelling developed introduction processed variational latent sensitivity scale restrictive adopted', 'algorithms approach popular insight greedy algorithmic maximization modularity results local', 'model classification data mixture missing outliers bayesian clustering method based', 'operators orders correspondence stack analog certain geometry unique calabi conjectural', 'log problems algorithm coloring lll pettie graphs round degree chung', 'rounds distributed best include solved currently revelation recent frugal conjecture', 'acoustic models layer normalization neural dln dynamically speakers technique deep', 'sphere time modelling line obstacle weather peron characterisation raised long', 'time model study question processes computing locations strategy asks bounds', 'modularity maximization algorithmic greedy insight popular community sparsity detection update', 'propagation susceptibility method based proposed belief combining palmer inverse concept', 'infty 

Training FASTopic: 100%|██████████| 92/92 [00:01<00:00, 51.63it/s] 
[I 2025-12-05 19:09:55,136] Trial 19 finished with value: -1000000.0 and parameters: {'num_topics': 39, 'DT_alpha': 1.0, 'normalize_embeddings': True, 'learning_rate': 0.0010519095028133836, 'epochs': 92}. Best is trial 0 with value: -1000000.0.


Top words found: ['linear cases identification double operation twisted threefolds ads form geometry', 'brandt recent showed mathsf solving defective lemma frugal revelation essentially', 'classification missing mixture outliers sensitivity clustering bayesian gives adopted values', 'memory traveling probability appear general path collects formalizing objectives graph', 'allow focuses approximation data latent learning possibility developed variational introduction', 'labeling solved providing automatically significant known conjectured classic checkable conjecture', 'understood performance metastable algorithms mechanism rule fails typically counting maximization', 'collected discrete potential goal encode strategies average maximizes accumulated existing', 'polynomial zygmund maximal lies truncations continuous define multidimensional operator obtained', 'sphere covariances weather type positive motivation extending obstacle extensions menegatto', 'degree coloring round pettie chung

In [16]:

# ========== Reentrenar el mejor modelo con los mejores hiperparámetros ==========
best = study.best_params
print("Reentrenando FASTopic con:", best)

# Intentar crear modelo con esos parámetros

best_model = FASTopic(num_topics=best["num_topics"],
                      normalize_embeddings=best.get("normalize_embeddings", True),
                      DT_alpha=best.get("DT_alpha", 5.0),
                      low_memory=best.get("low_memory", False),
                      device=best.get("device", "cpu"),
                      verbose=True)


# Fit final (intentar con lr/epochs)

best_top_words, best_doc_topic_dist = best_model.fit_transform(clean_text, learning_rate=best.get("learning_rate", 0.01), epochs=best.get("epochs", 100))


2025-12-05 19:11:03,093 - FASTopic - use device: cpu
2025-12-05 19:11:03,094 - FASTopic - First fit the model.


Reentrenando FASTopic con: {'num_topics': 36, 'DT_alpha': 1.0, 'normalize_embeddings': True, 'learning_rate': 0.05399484409787434, 'epochs': 128}


parsing texts: 100%|██████████| 10/10 [00:00<00:00, 3988.50it/s]
C:\Users\zeldan\anaconda3\envs\tfg-topics2\Lib\site-packages\sklearn\feature_extraction\text.py:517: UserWarning:

The parameter 'token_pattern' will not be used since 'tokenizer' is not None'

2025-12-05 19:11:04,636 - TopMost - Real vocab size: 498
2025-12-05 19:11:04,636 - TopMost - Real training size: 10 	 avg length: 79.300


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Training FASTopic: 100%|██████████| 128/128 [00:12<00:00, 10.07it/s]

Topic 0: code pulsedyn nonlinear ulam tsingou toda soliton systems body lennard dynamics specialists pasta integrable accessible
Topic 1: theorem mathbf mathbb polynomial carleson bounded kernel zygmund phases continuous maximal standard operator modulating lies
Topic 2: algebra theory omega koszul shown presented supergravity background branes explicitly orders supersymmetric dual duality gravitational
Topic 3: non numerical obtained fermi decay morse accessible black examine equilibrium recurrence source dynamical distribute quasi
Topic 4: log coloring round pettie number known randomized chung lcl making prove algorithm results recent defective
Topic 5: optimization formalizing modeled bounds average maximizes establish collected total goal given asks takes appear according
Topic 6: question sphere type extensions products extending schoenberg porcu prediction cross survey position similarly goes berg
Topic 7: modelling covariances obstacle geostatistical peron authors menegatto lon

In [17]:
best_model.get_top_words()

Topic 0: code pulsedyn nonlinear ulam tsingou toda soliton systems body lennard dynamics specialists pasta integrable accessible
Topic 1: theorem mathbf mathbb polynomial carleson bounded kernel zygmund phases continuous maximal standard operator modulating lies
Topic 2: algebra theory omega koszul shown presented supergravity background branes explicitly orders supersymmetric dual duality gravitational
Topic 3: non numerical obtained fermi decay morse accessible black examine equilibrium recurrence source dynamical distribute quasi
Topic 4: log coloring round pettie number known randomized chung lcl making prove algorithm results recent defective
Topic 5: optimization formalizing modeled bounds average maximizes establish collected total goal given asks takes appear according
Topic 6: question sphere type extensions products extending schoenberg porcu prediction cross survey position similarly goes berg
Topic 7: modelling covariances obstacle geostatistical peron authors menegatto lon

['code pulsedyn nonlinear ulam tsingou toda soliton systems body lennard dynamics specialists pasta integrable accessible',
 'theorem mathbf mathbb polynomial carleson bounded kernel zygmund phases continuous maximal standard operator modulating lies',
 'algebra theory omega koszul shown presented supergravity background branes explicitly orders supersymmetric dual duality gravitational',
 'non numerical obtained fermi decay morse accessible black examine equilibrium recurrence source dynamical distribute quasi',
 'log coloring round pettie number known randomized chung lcl making prove algorithm results recent defective',
 'optimization formalizing modeled bounds average maximizes establish collected total goal given asks takes appear according',
 'question sphere type extensions products extending schoenberg porcu prediction cross survey position similarly goes berg',
 'modelling covariances obstacle geostatistical peron authors menegatto long functions earth guella lines follows ber

In [18]:
fig = best_model.visualize_topic_hierarchy()
fig.show()

C:\Users\zeldan\anaconda3\envs\tfg-topics2\Lib\site-packages\fastopic\_plot.py:265: ClusterWarning:

The symmetric non-negative hollow observation matrix looks suspiciously like an uncondensed distance matrix



In [19]:
df_topics = pd.DataFrame({
    "topic_id":  range(len(best_top_words)),
    "top_words": ["".join(ws) for ws in best_top_words]
})
display(df_topics.head(40))

,topic_id,top_words
0,0,code pulsedyn nonlinear ulam tsingou toda soli...
1,1,theorem mathbf mathbb polynomial carleson boun...
2,2,algebra theory omega koszul shown presented su...
3,3,non numerical obtained fermi decay morse acces...
4,4,log coloring round pettie number known randomi...
5,5,optimization formalizing modeled bounds averag...
6,6,question sphere type extensions products exten...
7,7,modelling covariances obstacle geostatistical ...
8,8,carleson mathbb polynomial theorem mathbf boun...
9,9,results provides students single introduce sim...


In [20]:
best_model.visualize_topic(top_n=8)